# Exercise 6: Deployment & Serving on OpenShift

In this exercise, you will deploy your fine-tuned LLM to OpenShift/Kubernetes using vLLM for optimized serving.

## Learning Objectives

By the end of this exercise, you will be able to:
- Containerize a fine-tuned LLM with vLLM
- Configure Kubernetes deployment manifests for LLM serving
- Set appropriate resource limits and requests
- Test and validate the deployed LLM endpoint
- Understand LLM serving lifecycle management

## Prerequisites

Before starting this exercise, ensure you have:
1. Completed Exercise 5: Model Versioning & Packaging
2. A merged model available
3. Access to an OpenShift/Kubernetes cluster
4. The `oc` or `kubectl` CLI tools configured

---

## 1. Environment Setup

First, let's check our environment and review the artifacts we'll use for deployment.

In [ ]:
import os
import subprocess
import yaml
import json
from getpass import getpass

# Get Hugging Face token (needed if rebuilding image with private models)
hf_token = os.getenv("HF_TOKEN", None)
if hf_token:
    print("HF Token found")
else:
    print("Warning: No HF Token provided. You may encounter rate limits.")

# Check kubectl/oc availability
def check_cli_tool(name):
    try:
        result = subprocess.run([name, 'version', '--client'], capture_output=True, text=True, timeout=5)
        print(f"{name}: available")
        return True
    except (FileNotFoundError, subprocess.TimeoutExpired):
        print(f"{name}: not found")
        return False

has_kubectl = check_cli_tool('kubectl')
has_oc = check_cli_tool('oc')

# Check if connected to a cluster
if has_oc:
    result = subprocess.run(['oc', 'whoami'], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print(f"Connected as: {result.stdout.strip()}")
    else:
        print("Not connected to any cluster. Deployment commands will be for reference only.")
elif has_kubectl:
    result = subprocess.run(['kubectl', 'config', 'current-context'], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print(f"Current context: {result.stdout.strip()}")
    else:
        print("Not connected to any cluster.")


## 2. Review the Deployment Manifests

Let's examine the Kubernetes manifests we'll use for deployment.

In [ ]:
# Read and display the deployment manifest
k8s_dir = os.path.join('..', 'k8s')

print("=" * 60)
print("Deployment Manifest")
print("=" * 60)
with open(os.path.join(k8s_dir, 'deployment.yaml'), 'r') as f:
    print(f.read())

print("=" * 60)
print("Service Manifest")
print("=" * 60)
with open(os.path.join(k8s_dir, 'service.yaml'), 'r') as f:
    print(f.read())

### Understanding the Deployment Configuration

**Resource Management**:
- Requests: 8Gi memory, 2 CPUs (guaranteed minimum)
- Limits: 16Gi memory, 4 CPUs (burst cap)

**Health Probes**:
- Readiness: checks `/health` after 60s, every 10s
- Liveness: checks `/health` after 120s, every 20s

**Why these values?** LLM serving is memory-intensive. TinyLlama-1.1B in half precision needs ~2.2GB for weights, plus ~4-8GB for KV cache during inference.

## 3. Build and Push the Container Image

Let's review the Dockerfile and build the container image for deployment.

In [ ]:
# Review the Dockerfile
print("=" * 60)
print("Dockerfile")
print("=" * 60)
with open(os.path.join('..', 'Dockerfile'), 'r') as f:
    print(f.read())

### Build Commands

Run the following commands in your terminal to build and push the image:

```bash
# Build the Docker image
docker build -t llm-instruction-tuning:latest .

# Tag for your registry
docker tag llm-instruction-tuning:latest <your-registry>/llm-instruction-tuning:latest

# Push to registry
docker push <your-registry>/llm-instruction-tuning:latest
```

For OpenShift AI, use the internal registry:
```bash
# Login to OpenShift
oc login <openshift-cluster>

# Build and push
docker build -t <image-registry-path>/llm-instruction-tuning:latest .
docker push <image-registry-path>/llm-instruction-tuning:latest
```

## 4. Deploy to OpenShift/Kubernetes

Now let's deploy our LLM serving application to the cluster.

In [ ]:
# Generate deployment commands
print("Run these commands in your terminal to deploy:\n")

print("# Create a namespace for the workshop")
print("oc new-project llm-workshop --display-name='LLM Instruction Tuning Workshop'\n")

print("# Update the image in deployment.yaml first")
print("# Edit k8s/deployment.yaml to set image: <your-registry>/llm-instruction-tuning:latest\n")

print("# Apply manifests")
print("oc apply -f k8s/deployment.yaml")
print("oc apply -f k8s/service.yaml\n")

print("# Check status")
print("oc get deployments")
print("oc get pods")
print("oc get services")

## 5. Access the LLM Endpoint

Once the deployment is running, access the model via port forwarding or a route.

In [ ]:
print("Option 1: Port Forwarding (for testing)")
print("oc port-forward service/llm-instruction-tuning-service 8000:80")
print("# API available at: http://localhost:8000/v1/chat/completions\n")

print("Option 2: Create a Route (OpenShift)")
print("oc expose svc/llm-instruction-tuning-service")
print("# Get the route URL:")
print("oc get route llm-instruction-tuning-service\n")

print("Option 3: Ingress (Kubernetes)")
ingress_yaml = {
    "apiVersion": "networking.k8s.io/v1",
    "kind": "Ingress",
    "metadata": {"name": "llm-instruction-tuning-ingress"},
    "spec": {
        "rules": [{
            "host": "llm-workshop.example.com",
            "http": {
                "paths": [{
                    "path": "/",
                    "pathType": "Prefix",
                    "backend": {
                        "service": {
                            "name": "llm-instruction-tuning-service",
                            "port": {"number": 80}
                        }
                    }
                }]
            }
        }]
    }
}
print(yaml.dump(ingress_yaml, default_flow_style=False))

## 6. Test the Deployed LLM

Let's test our deployed model using curl and Python.

In [ ]:
# Test endpoint configuration
ENDPOINT = "http://localhost:8000/v1/chat/completions"  # Update if using route
MODEL = "TinyLlama-1.1B-Chat-v1.0"

# Test with curl equivalent using Python requests
import requests

def test_llm_endpoint(endpoint, model, prompt, max_tokens=100):
    """Test the LLM serving endpoint."""
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": 0.7
    }
    headers = {"Content-Type": "application/json"}

    try:
        response = requests.post(endpoint, json=payload, headers=headers, timeout=30)
        if response.status_code == 200:
            result = response.json()
            return result['choices'][0]['message']['content']
        else:
            return f"Error: {response.status_code} - {response.text}"
    except requests.exceptions.ConnectionError:
        return "Connection refused. Is the server running?"
    except Exception as e:
        return f"Error: {e}"

# Test with sample prompts
test_prompts = [
    "What is MLOps?",
    "Explain the difference between CPU and GPU.",
    "Write a short poem about data science."
]

print("Testing LLM Endpoint")
print("=" * 50)
print(f"Endpoint: {ENDPOINT}")
print(f"Model: {MODEL}")
print("=" * 50)

for i, prompt in enumerate(test_prompts):
    print(f"\nPrompt {i+1}: {prompt}")
    response = test_llm_endpoint(ENDPOINT, MODEL, prompt)
    print(f"Response: {response}\n")

### Test with the Workshop Test Client

You can also use the dedicated test client script:

```bash
# Test with port forwarding
python scripts/test_client.py \
    --endpoint http://localhost:8000/v1/chat/completions \
    --model TinyLlama-1.1B-Chat-v1.0 \
    --prompt "Explain MLOps in simple terms." \
    --max-tokens 100 \
    --num-requests 3 \
    --stream
```

## 7. Monitor and Manage the Deployment

Let's explore monitoring and management commands for the deployed LLM.

In [ ]:
# Generate management commands
print("Management Commands")
print("=" * 50)

print("\n# View logs")
print("oc logs -f deployment/llm-instruction-tuning-deployment")

print("\n# Scale the deployment")
print("oc scale deployment/llm-instruction-tuning-deployment --replicas=2")

print("\n# Check resource usage")
print("oc top pods")

print("\n# Describe deployment for details")
print("oc describe deployment/llm-instruction-tuning-deployment")

print("\n# Update with new model version")
print("# 1. Build new image with updated model")
print("# 2. Update image tag in k8s/deployment.yaml")
print("# 3. Apply the update:")
print("oc apply -f k8s/deployment.yaml")

print("\n# Rollback if needed")
print("oc rollout undo deployment/llm-instruction-tuning-deployment")

print("\n# Delete deployment")
print("oc delete -f k8s/deployment.yaml")
print("oc delete -f k8s/service.yaml")

## 8. LLM Serving Lifecycle

Understanding the full lifecycle of an LLM serving deployment:

| Phase | Description | Key Actions |
|-------|-------------|-------------|
| **1. Build** | Containerize model + dependencies | `docker build`, `docker push` |
| **2. Deploy** | Roll out to cluster | `oc apply -f k8s/deployment.yaml` |
| **3. Verify** | Health checks, test inference | `/health`, test prompts |
| **4. Expose** | Make accessible via network | Service, Route, Ingress |
| **5. Scale** | Adjust capacity | `oc scale --replicas=N` |
| **6. Update** | Roll out new model version | Update image, rolling update |
| **7. Monitor** | Track metrics, logs | `oc logs`, `oc top`, dashboards |
| **8. Rollback** | Revert to previous version | `oc rollout undo` |
| **9. Decommission** | Remove old deployments | `oc delete -f k8s/` |

### Resource Considerations for LLM Serving

| Factor | Impact | Recommendation |
|--------|--------|----------------|
| Model size | Memory for weights | 4x model params in bytes for FP16 |
| Batch size | Memory for KV cache | ~2-8GB per concurrent request |
| Sequence length | Memory per token | Longer sequences = more KV cache |
| Concurrent users | Total memory needed | batch_size * memory_per_request |
| GPU availability | Inference speed | 10-100x faster than CPU |

In [ ]:
# Memory estimation for TinyLlama serving
model_params = 1.1  # billion
fp16_bytes_per_param = 2  # bytes

weight_memory_gb = model_params * fp16_bytes_per_param / 1.024
kv_cache_per_request_gb = 2.0  # estimated
concurrent_users = 4
overhead_gb = 1.0

total_memory_gb = (weight_memory_gb +
                   kv_cache_per_request_gb * concurrent_users +
                   overhead_gb)

print("Memory Estimation for TinyLlama-1.1B Serving")
print("=" * 50)
print(f"Model weights (FP16): {weight_memory_gb:.1f} GB")
print(f"KV cache ({concurrent_users} concurrent): {kv_cache_per_request_gb * concurrent_users:.1f} GB")
print(f"Overhead: {overhead_gb:.1f} GB")
print(f"Total estimated memory: {total_memory_gb:.1f} GB")
print()
print(f"Recommended memory limit: 16 Gi (as configured in deployment.yaml)")
print(f"This provides headroom for peak workloads and cold start.")

## 9. Deployment Checklist

Before considering your deployment production-ready, verify each item:

- [ ] Container image built and pushed to registry
- [ ] Image tag updated in `k8s/deployment.yaml`
- [ ] Resource requests/limits set appropriately
- [ ] Health probes configured
- [ ] Service created and accessible
- [ ] Route or Ingress configured for external access
- [ ] Model responds correctly to test prompts
- [ ] Logs show no errors
- [ ] Resource usage within configured limits
- [ ] Rollback procedure documented

## Summary

In this exercise, we learned how to:
1. Containerize a fine-tuned LLM with vLLM for optimized serving
2. Review and apply Kubernetes deployment and service manifests
3. Build and push the container image to a registry
4. Deploy to OpenShift/Kubernetes with proper resource management
5. Access and test the deployed LLM endpoint
6. Monitor and manage the LLM serving lifecycle
7. Estimate memory requirements for LLM serving

**Key Takeaways:**
- LLM serving requires careful resource planning (memory is the bottleneck)
- Kubernetes health probes are critical for long-startup applications
- vLLM provides optimized inference with continuous batching
- Model lifecycle management enables safe, rolling updates

---

*Congratulations on completing all workshop exercises!*